# 🧪 GIAI ĐOẠN 3 (NHÓM 2): TRANSFORMER ABLATION STUDY & SO SÁNH 3 BỘ TOKENIZER (GÓI GỌN 4 RUN)
**Đề tài 8: Dịch máy Anh – Việt (IWSLT'15 En–Vi) | Môn học: Học sâu**  
**Phụ trách: Sơn (Phần 3 – Nhóm trưởng)**  
---
> **Lưu ý:** Chọn GPU: **Runtime -> Change runtime type -> T4 GPU** và bật Colab Secret **`BTL_DL_COLAB`**.
>
> **Đã hoàn thành trước đó (Mốc Base):** `transformer_bpe_base` (`N=4, H=8, PE=True, LS=0.1`) — Greedy `25.76` / Beam(k=4) **`26.71 BLEU`**.
>
> **Các cải tiến chống ngắt/mất file trong bản này:**
> - **Tắt hoàn toàn spam log `tqdm`** (`TQDM_DISABLE=1` + lọc dòng `%|`), chỉ in tóm tắt từng Epoch và bảng BLEU cuối.
> - **Lưu checkpoint trực tiếp lên Google Drive** ngay khi train (Colab ngắt giữa chừng vẫn không mất `best_model.pt`; chạy lại sẽ tự `[RESUME]` nhảy thẳng sang `eval.py`).
> - Rút gọn còn **4 run cốt lõi** (`epochs=15`, chỉ eval `beam_size=4, alpha=0.6`) — đủ 100% bảng **3 Tokenizer** + bảng **Ablation Study** cho báo cáo ACL:
>   1. `transformer_bpe_L2H8` *(Ablation: giảm số tầng `N=2`)*
>   2. `transformer_bpe_noPE` *(Ablation: tắt Positional Encoding)*
>   3. `transformer_word_base` *(Tokenizer: Word-level)*
>   4. `transformer_spm_base` *(Tokenizer: SentencePiece)*

### Bước 1: Kiểm tra GPU & Mount Google Drive chung

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

### Bước 2: Clone / Cập nhật mã nguồn nhánh `main` & Cài đặt thư viện

In [ ]:
import os

REPO_URL = "https://github.com/hoang180/Machine_Translation_Project.git"
PROJECT_DIR = "/content/Machine_Translation_Project"
BRANCH = "main"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("✓ Đã sẵn sàng mã nguồn nhánh main và thư viện!")

### Bước 3: Copy dữ liệu `data/processed/` & 3 bộ Tokenizer từ Google Drive chung (`BTL_DL_Translation`)

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/BTL_DL_Translation"

!mkdir -p data/processed custom_tokenizers/word custom_tokenizers/bpe custom_tokenizers/spm
!cp -r "{DRIVE_ROOT}/data/processed/." data/processed/
!cp -r "{DRIVE_ROOT}/custom_tokenizers/." custom_tokenizers/

!ls -lh data/processed/ custom_tokenizers/word/ custom_tokenizers/bpe/ custom_tokenizers/spm/

### Bước 4: Auto-login Weights & Biases (`btl-dl-translation-iwslt15`)

In [ ]:
import os
import wandb
from google.colab import userdata

api_key = userdata.get("BTL_DL_COLAB")
if api_key:
    os.environ["WANDB_API_KEY"] = api_key
    wandb.login(key=api_key, relogin=True)
else:
    wandb.login(relogin=True)

api = wandb.Api()
print("✓ Đang đăng nhập W&B:", api.viewer.username, "| Default entity:", api.default_entity)

### Bước 5: Định nghĩa hàm `run_experiment()` (Không spam log + Lưu thẳng lên Drive + Tự resume)

In [ ]:
import json
import os
import re
import subprocess

DRIVE_ROOT = "/content/drive/MyDrive/BTL_DL_Translation"


def _run_clean(cmd: list[str]) -> str:
    """Chạy lệnh subprocess, tắt hẳn thanh tqdm và lọc bỏ mọi dòng progress bar gây tràn output Colab."""
    print("\n>>>", " ".join(cmd), flush=True)
    env = os.environ.copy()
    env["TQDM_DISABLE"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    kept_lines = []
    for line in proc.stdout:
        # Bỏ qua hoàn toàn mọi dòng thanh tiến trình tqdm/wandb spinner nếu còn sót
        if "%|" in line or "it/s]" in line or "s/it]" in line:
            continue
        if "Waiting for wandb.init" in line:
            continue
        print(line, end="", flush=True)
        kept_lines.append(line)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {ret}): {' '.join(cmd)}")
    return "".join(kept_lines)


def _parse_bleu(output: str) -> dict:
    res = {}
    m = re.search(r"Điểm BLEU:\s*([0-9.]+)", output)
    if m:
        res["bleu"] = float(m.group(1))
    m_bp = re.search(r"Brevity Penalty:\s*([0-9.]+)", output)
    if m_bp:
        res["bp"] = float(m_bp.group(1))
    m_sig = re.search(r"Signature:\s*(.+)", output)
    if m_sig:
        res["signature"] = m_sig.group(1).strip()
    return res


def run_experiment(
    exp_dir_name: str,
    tokenizer: str = "bpe",
    num_layers: int = 4,
    num_heads: int = 8,
    use_pe: bool = True,
    smoothing: float = 0.1,
    epochs: int = 15,
    batch_size: int = 128,
    skip_if_done: bool = True,
):
    # Lưu thẳng lên Google Drive ngay trong lúc train -> KHÔNG bao giờ mất checkpoint nếu Colab reset
    drive_dir = os.path.join(DRIVE_ROOT, "checkpoints", exp_dir_name)
    ckpt_path = os.path.join(drive_dir, "best_model.pt")
    summary_path = os.path.join(drive_dir, "train_summary.json")
    drive_eval_file = os.path.join(drive_dir, "eval_result.json")

    if skip_if_done and os.path.exists(drive_eval_file):
        print(f"[SKIP] {exp_dir_name} đã hoàn thành trên Drive: {drive_eval_file}")
        with open(drive_eval_file, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), ensure_ascii=False, indent=2))
        return

    os.makedirs(drive_dir, exist_ok=True)

    # 1. Train Transformer (tự bỏ qua nếu checkpoint + summary đã có trên Drive)
    if skip_if_done and os.path.exists(ckpt_path) and os.path.exists(summary_path):
        print(f"[RESUME] Đã thấy checkpoint & summary của {exp_dir_name} trên Drive -> bỏ qua train, chạy thẳng eval.py!")
    else:
        _run_clean([
            "python3", "train_transformer.py",
            "--config", "configs/transformer_config.yaml",
            "--tokenizer", tokenizer,
            "--num_layers", str(num_layers),
            "--num_heads", str(num_heads),
            "--use_pe", "true" if use_pe else "false",
            "--smoothing", str(smoothing),
            "--epochs", str(epochs),
            "--batch_size", str(batch_size),
            "--save_dir", drive_dir,
        ])

    # 2. Eval nhanh bằng Beam Search (k=4, alpha=0.6) trên tst2013 (1,268 câu)
    print(f"\n[INFO] Đang chạy eval.py (beam_size=4) trên tst2013 cho {exp_dir_name} (không in progress bar, chờ ~3-5 phút)...", flush=True)
    out_beam4 = _run_clean([
        "python3", "eval.py", "--checkpoint", ckpt_path, "--beam_size", "4", "--alpha", "0.6"
    ])

    eval_result = {
        "exp_dir": exp_dir_name,
        "tokenizer": tokenizer,
        "num_layers": num_layers,
        "num_heads": num_heads,
        "use_pe": use_pe,
        "smoothing": smoothing,
        "beam4": _parse_bleu(out_beam4),
    }
    with open(drive_eval_file, "w", encoding="utf-8") as f:
        json.dump(eval_result, f, ensure_ascii=False, indent=2)

    print(f"\n✓ HOÀN TẤT & ĐÃ LƯU DRIVE: {drive_dir}")
    print(f"  -> Test BLEU (Beam k=4, alpha=0.6): {eval_result['beam4'].get('bleu')}")

print("✓ Đã sẵn sàng hàm run_experiment() (chống spam log + lưu thẳng Drive)!")

---
## PHẦN A: 2 BIẾN THỂ ABLATION CỐT LÕI (TRÊN BPE TOKENIZER)

### Run 1/4 — Ablation độ sâu: Giảm số tầng (`N=2, H=8`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_bpe_L2H8",
    tokenizer="bpe",
    num_layers=2,
    num_heads=8,
    use_pe=True,
    smoothing=0.1,
    epochs=15,
    batch_size=128,
)

### Run 2/4 — Ablation Positional Encoding: Tắt PE (`use_pe=False`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_bpe_noPE",
    tokenizer="bpe",
    num_layers=4,
    num_heads=8,
    use_pe=False,
    smoothing=0.1,
    epochs=15,
    batch_size=128,
)

---
## PHẦN B: 2 BỘ TOKENIZER CÒN LẠI TRÊN TRANSFORMER (`Word` & `SentencePiece`)

### Run 3/4 — Tokenizer Word-level (`transformer_word_base`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_word_base",
    tokenizer="word",
    num_layers=4,
    num_heads=8,
    use_pe=True,
    smoothing=0.1,
    epochs=15,
    batch_size=128,
)

### Run 4/4 — Tokenizer SentencePiece (`transformer_spm_base`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_spm_base",
    tokenizer="spm",
    num_layers=4,
    num_heads=8,
    use_pe=True,
    smoothing=0.1,
    epochs=15,
    batch_size=128,
)

---
### *(Tùy chọn nếu còn dư quota GPU)* — 2 biến thể Ablation bổ sung (`H=4` và tắt Label Smoothing)

In [ ]:
# Bỏ dấu # nếu muốn chạy thêm 2 biến thể phụ:
# run_experiment("transformer_bpe_L4H4", tokenizer="bpe", num_layers=4, num_heads=4, use_pe=True, smoothing=0.1, epochs=15)
# run_experiment("transformer_bpe_noLS", tokenizer="bpe", num_layers=4, num_heads=8, use_pe=True, smoothing=0.0, epochs=15)

---
## PHẦN C: TỔNG HỢP BẢNG KẾT QUẢ CSV & JSON (CHÈN VÀO BÁO CÁO ACL)

In [ ]:
import json
import os
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/BTL_DL_Translation"
CKPT_ROOT = os.path.join(DRIVE_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

EXPERIMENTS = [
    ("transformer_bpe_base",  "Base (BPE, N=4, H=8, PE, LS=0.1)"),
    ("transformer_bpe_L2H8",  "Ablation: N=2 layers (H=8)"),
    ("transformer_bpe_noPE",  "Ablation: No Positional Encoding"),
    ("transformer_bpe_L4H4",  "Ablation: H=4 heads (N=4)"),
    ("transformer_bpe_noLS",  "Ablation: No Label Smoothing (0.0)"),
    ("transformer_word_base", "Tokenizer: Word-level (N=4, H=8)"),
    ("transformer_spm_base",  "Tokenizer: SentencePiece (N=4, H=8)"),
]

# Đảm bảo transformer_bpe_base (đã chạy ở notebook 02) có eval_result.json
base_eval_path = os.path.join(CKPT_ROOT, "transformer_bpe_base", "eval_result.json")
if not os.path.exists(base_eval_path) and os.path.exists(os.path.join(CKPT_ROOT, "transformer_bpe_base")):
    with open(base_eval_path, "w", encoding="utf-8") as f:
        json.dump({
            "exp_dir": "transformer_bpe_base",
            "tokenizer": "bpe",
            "num_layers": 4,
            "num_heads": 8,
            "use_pe": True,
            "smoothing": 0.1,
            "beam4": {"bleu": 26.71, "bp": 0.9429},
        }, f, ensure_ascii=False, indent=2)

rows = []
for folder, label in EXPERIMENTS:
    s_path = os.path.join(CKPT_ROOT, folder, "train_summary.json")
    e_path = os.path.join(CKPT_ROOT, folder, "eval_result.json")
    if not os.path.exists(s_path):
        continue
    with open(s_path, "r", encoding="utf-8") as f:
        s = json.load(f)
    ev = json.load(open(e_path, "r", encoding="utf-8")) if os.path.exists(e_path) else {}
    m_cfg = s.get("config", {}).get("model", {})
    t_cfg = s.get("config", {}).get("training", {})
    best_ep = min(s.get("history", []), key=lambda r: r["val/loss"], default={})
    rows.append({
        "Config": label,
        "Run": s.get("run_name", folder),
        "Tokenizer": s.get("tokenizer_type"),
        "Layers": m_cfg.get("num_layers"),
        "Heads": m_cfg.get("num_heads"),
        "PE": m_cfg.get("use_pe"),
        "LS": t_cfg.get("label_smoothing"),
        "Params": s.get("trainable_params"),
        "Avg_Epoch_s": s.get("avg_epoch_time_sec"),
        "Best_Epoch": best_ep.get("epoch"),
        "Val_Loss": s.get("best_val_loss"),
        "Val_PPL": best_ep.get("val/perplexity"),
        "Test_BLEU_Beam4": ev.get("beam4", {}).get("bleu"),
    })

df = pd.DataFrame(rows)
csv_out = os.path.join(RESULTS_DIR, "transformer_experiments_summary.csv")
json_out = os.path.join(RESULTS_DIR, "transformer_experiments_summary.json")
df.to_csv(csv_out, index=False, encoding="utf-8")
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"✓ Đã xuất bảng tổng hợp ({len(rows)} runs) ra:\n  - {csv_out}\n  - {json_out}\n")
print(df.to_markdown(index=False))